# Aplicar cargas de flexibilidad a un modelo SAP2000 (.s2k)

**Entradas** (datasets de Kaggle):
- `.s2k` base exportado desde SAP2000, con los estados de carga ya creados como load patterns.
- Excel/CSV por bloques: en la columna A el nodo de flexibilidad (cabecera de bloque), en la B el estado de carga, y en C, D, E las cargas X, Y, Z (kgf).

En SAP, cada nodo de flexibilidad debe estar en un grupo con el mismo nombre (por ejemplo `10072` o `N 10072`).

**Reglas:**
- Se aplican solo los nodos que tienen grupo con un único joint y los estados que existen en el `.s2k`.
- Lo que falte (nodos sin grupo, estados no creados) se salta y se lista en la salida.
- Un grupo con más de un joint sí detiene la ejecución, porque no se sabe a qué joint aplicar la carga.

**Salida:** `/kaggle/working/<modelo>_CARGAS.s2k`. Cada ejecución parte del `.s2k` base, así que puedes repetirla con Excel modificados.

In [ ]:
import re
import glob
from pathlib import Path

import pandas as pd

INPUT_DIR = "/kaggle/input"
OUTPUT_DIR = "/kaggle/working"

MODEL_FILE = None   # ruta fija al .s2k; None = detectar automáticamente
LOADS_FILE = None   # ruta fija al Excel/CSV; None = detectar automáticamente

SIGN = 1.0                       # -1.0 si las cargas de flexibilidad tienen el signo opuesto al de SAP
LOAD_CS = "GLOBAL"               # sistema de coordenadas de las cargas
ENCODING = "latin-1"             # preserva los bytes del .s2k al reescribirlo


In [ ]:
def find_single(exts):
    hits = sorted(p for p in glob.glob(f"{INPUT_DIR}/**/*", recursive=True)
                  if Path(p).suffix.lower() in exts)
    if len(hits) != 1:
        raise FileNotFoundError(
            f"Se esperaba un archivo {sorted(exts)} en {INPUT_DIR}, encontrados: {hits}. "
            "Fija MODEL_FILE / LOADS_FILE en la celda de configuración."
        )
    return hits[0]


model_path = MODEL_FILE or find_single({".s2k"})
if Path(model_path).stem.endswith("_CARGAS"):
    raise ValueError("El .s2k de entrada es una salida de este notebook. Usa siempre el modelo base: cada ejecución parte de él.")
loads_path = LOADS_FILE or find_single({".xlsx", ".xls", ".csv"})
print("Modelo:", model_path)
print("Cargas:", loads_path)


In [ ]:
def normalize_name(v):
    s = str(v).strip()
    return s[:-2] if re.fullmatch(r"\d+\.0", s) else s


def read_loads(path):
    """Formato por bloques: la columna A trae el nodo en la fila de cabecera del bloque,
    la columna B el estado y las columnas C, D, E las cargas X, Y, Z.
    Las filas sin estado (cabeceras, MAX, vacías) se saltan. Una tabla plana también funciona."""
    if path.lower().endswith(".csv"):
        raw = pd.read_csv(path, header=None, dtype=str)
    else:
        raw = pd.read_excel(path, header=None, dtype=str)
    raw = raw.iloc[:, :5]
    raw.columns = ["node", "state", "X", "Y", "Z"]

    rows, node = [], None
    for i, r in raw.iterrows():
        if pd.notna(r["node"]):
            node = normalize_name(r["node"])
        if pd.isna(r["state"]) or str(r["state"]).strip().upper() == "MAX":
            continue
        if node is None:
            raise ValueError(f"Fila {i + 1}: hay un estado sin nodo encima.")
        rows.append({"node": node, "state": normalize_name(r["state"]),
                     "X": r["X"], "Y": r["Y"], "Z": r["Z"]})

    df = pd.DataFrame(rows, columns=["node", "state", "X", "Y", "Z"])
    for c in ["X", "Y", "Z"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    bad = df[df[["X", "Y", "Z"]].isna().any(axis=1)]
    if len(bad):
        print(f"Ignoradas {len(bad)} filas sin valor numérico en X/Y/Z:")
        print(bad.to_string())
    df = df.dropna(subset=["X", "Y", "Z"]).reset_index(drop=True)

    dup = df[df.duplicated(["node", "state"], keep=False)]
    if len(dup):
        raise ValueError(f"Filas duplicadas (nodo, estado):\n{dup.to_string()}")
    return df


loads_df = read_loads(loads_path)
print(f"{len(loads_df)} filas, {loads_df['node'].nunique()} nodos, {loads_df['state'].nunique()} estados")
loads_df.head()


In [ ]:
TABLE_RE = re.compile(r'^TABLE:\s+"(.+?)"')
KV_RE = re.compile(r'(\w+)=("[^"]*"|\S+)')


def parse_s2k(path):
    with open(path, encoding=ENCODING) as f:
        lines = f.read().splitlines()
    tables = {}
    current = None
    for line in lines:
        m = TABLE_RE.match(line.strip())
        if m:
            current = m.group(1)
            tables.setdefault(current, [])
        elif current is not None and "=" in line:
            tables[current].append({k: v.strip('"') for k, v in KV_RE.findall(line)})
    return lines, tables


def group_members(tables):
    """Devuelve {grupo: {joints}} a partir de las filas de asignación de grupos."""
    members = {}
    for rows in tables.values():
        for row in rows:
            low = {k.lower(): v for k, v in row.items()}
            group = low.get("group") or low.get("groupname")
            joint = low.get("joint")
            if joint is None and low.get("objecttype", "").lower() == "joint":
                joint = low.get("objectlabel")
            if group and joint:
                members.setdefault(group, set()).add(joint)
    return members


lines, tables = parse_s2k(model_path)
print("Tablas con 'GROUP' en el nombre:")
for name, rows in tables.items():
    if "GROUP" in name.upper():
        print(f"  {name}: {len(rows)} filas  ej.: {rows[:2]}")

units = next((r["CurrUnits"] for r in tables.get("PROGRAM CONTROL", []) if "CurrUnits" in r), None)
if units is None:
    print("Aviso: no encontré CurrUnits en PROGRAM CONTROL; verifica a mano que el modelo esté en kgf-m.")
else:
    parts = [p.strip().lower() for p in units.split(",")]
    if parts[:2] != ["kgf", "m"]:
        raise ValueError(f"El modelo está en '{units}', no en kgf-m. Cambia unidades en SAP o ajusta el Excel.")
    print("Unidades del modelo:", units)

members = group_members(tables)
if not members:
    raise ValueError("No encontré asignaciones de grupo a joints en el .s2k. Revisa la salida de arriba.")
print(f"\n{len(members)} grupos con joints asignados")


In [ ]:
def match_nodes(df, members):
    """Empareja cada nodo con su grupo de SAP. Los nodos sin grupo se saltan y se listan."""
    lookup = {g.lower(): g for g in members}
    mapping, sin_grupo, multi = {}, [], []
    for node in df["node"].unique():
        found = None
        for cand in (node, f"N {node}", f"N{node}"):
            if cand.lower() in lookup:
                found = lookup[cand.lower()]
                break
        if found is None:
            sin_grupo.append(node)
            continue
        joints = sorted(members[found])
        if len(joints) != 1:
            multi.append(f"nodo {node}: grupo '{found}' tiene {len(joints)} joints {joints}")
            continue
        mapping[node] = joints[0]
    if multi:
        raise ValueError("Grupos con más de un joint (no se sabe a cuál aplicar la carga):\n  " + "\n  ".join(multi))
    print(f"Nodos con grupo: {len(mapping)}. Sin grupo (se saltan): {len(sin_grupo)}")
    if sin_grupo:
        print("  " + ", ".join(sin_grupo))
    return mapping


node_to_joint = match_nodes(loads_df, members)
print(pd.DataFrame(list(node_to_joint.items()), columns=["nodo_flex", "joint_sap"]).to_string(index=False))


In [ ]:
defined_patterns = {r["LoadPat"] for r in tables.get("LOAD PATTERN DEFINITIONS", []) if "LoadPat" in r}

n_total = len(loads_df)
to_apply = loads_df[loads_df["node"].isin(node_to_joint)].copy()
to_apply["joint"] = to_apply["node"].map(node_to_joint)

missing_states = sorted(set(to_apply["state"]) - defined_patterns)
if missing_states:
    print("Estados que no existen en el modelo (se saltan):", missing_states)
to_apply = to_apply[to_apply["state"].isin(defined_patterns)].reset_index(drop=True)

print(f"Cargas a aplicar: {len(to_apply)} de {n_total}")
if to_apply.empty:
    raise ValueError("No queda ninguna carga que aplicar: revisa nodos, grupos y nombres de estados.")


In [ ]:
def fmt(v):
    return f'"{v}"'  # siempre entre comillas: nombres como 1(OPE) o N 10072


def num(v):
    return f"{float(v) * SIGN:.10g}"


joint_rows = [
    f"   Joint={fmt(r.joint)}  LoadPat={fmt(r.state)}  CoordSys={LOAD_CS}  "
    f"F1={num(r.X)}  F2={num(r.Y)}  F3={num(r.Z)}  M1=0  M2=0  M3=0"
    for r in to_apply.itertuples()
]


def add_rows(lines, table_name, new_rows):
    """Inserta filas justo debajo de la cabecera de la tabla; si no existe, la crea antes de END TABLE DATA."""
    for i, line in enumerate(lines):
        m = TABLE_RE.match(line.strip())
        if m and m.group(1) == table_name:
            return lines[:i + 1] + new_rows + lines[i + 1:]
    block = ["", f'TABLE:  "{table_name}"'] + new_rows
    end = next((i for i, l in enumerate(lines) if l.strip() == "END TABLE DATA"), len(lines))
    return lines[:end] + block + lines[end:]


def drop_placeholder_loads(lines, states):
    """Quita las cargas de joint de cero que SAP crea al crear un estado. Si hay alguna distinta de cero, para."""
    keep, dropped, table = [], 0, None
    for line in lines:
        m = TABLE_RE.match(line.strip())
        if m:
            table = m.group(1)
        elif table == "JOINT LOADS - FORCE" and "=" in line:
            row = {k: v.strip('"') for k, v in KV_RE.findall(line)}
            if row.get("LoadPat") in states:
                if any(float(row.get(k, 0)) != 0 for k in ["F1", "F2", "F3", "M1", "M2", "M3"]):
                    raise ValueError(f"El modelo ya tiene una carga distinta de cero en {row.get('LoadPat')} (joint {row.get('Joint')}). Revísala en SAP.")
                dropped += 1
                continue
        keep.append(line)
    print(f"Cargas de cero previas eliminadas: {dropped}")
    return keep


lines = drop_placeholder_loads(lines, set(to_apply["state"]))
out_lines = add_rows(lines, "JOINT LOADS - FORCE", joint_rows)

out_path = Path(OUTPUT_DIR) / f"{Path(model_path).stem}_CARGAS.s2k"
out_path.parent.mkdir(parents=True, exist_ok=True)
with open(out_path, "w", encoding=ENCODING, newline="\n") as f:
    f.write("\n".join(out_lines) + "\n")
print(f"{len(joint_rows)} cargas de joint escritas en: {out_path}")


In [ ]:
import numpy as np

_, out_tables = parse_s2k(out_path)
new_loads = [r for r in out_tables["JOINT LOADS - FORCE"] if r.get("LoadPat") in set(to_apply["state"])]
assert len(new_loads) == len(to_apply), f"Esperadas {len(to_apply)} cargas, encontradas {len(new_loads)}"

check = pd.DataFrame(new_loads)
for c in ["F1", "F2", "F3"]:
    check[c] = check[c].astype(float)
got = check.groupby("LoadPat")[["F1", "F2", "F3"]].sum()
exp = to_apply.assign(X=to_apply.X * SIGN, Y=to_apply.Y * SIGN, Z=to_apply.Z * SIGN) \
              .groupby("state")[["X", "Y", "Z"]].sum()
diff = np.abs(got.values - exp.loc[got.index].values).max()
print("Máxima diferencia en sumas por estado:", diff)
assert diff < 1e-6, "Las sumas no cuadran"
print("Verificación OK")
got
